In [2]:
import requests

pokemon_list = []
from tqdm import tqdm

for i in tqdm(range(1, 11)):
    response = requests.get(f"https://pokeapi.co/api/v2/pokemon/{i}")
    if response.status_code == 200:
        pokemon_list.append(response.json())
    else:
        print(f"Failed to fetch Pokemon {i}")

# Display the number of Pokemon fetched
print(f"Fetched {len(pokemon_list)} Pokemon data entries.")

Fetched 10 Pokemon data entries.


In [6]:
pokemon_list[0].keys()

dict_keys(['abilities', 'base_experience', 'cries', 'forms', 'game_indices', 'height', 'held_items', 'id', 'is_default', 'location_area_encounters', 'moves', 'name', 'order', 'past_abilities', 'past_types', 'species', 'sprites', 'stats', 'types', 'weight'])

In [8]:
pokemon_list[0]["game_indices"]

[{'game_index': 153,
  'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}},
 {'game_index': 153,
  'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}},
 {'game_index': 153,
  'version': {'name': 'yellow',
   'url': 'https://pokeapi.co/api/v2/version/3/'}},
 {'game_index': 1,
  'version': {'name': 'gold', 'url': 'https://pokeapi.co/api/v2/version/4/'}},
 {'game_index': 1,
  'version': {'name': 'silver',
   'url': 'https://pokeapi.co/api/v2/version/5/'}},
 {'game_index': 1,
  'version': {'name': 'crystal',
   'url': 'https://pokeapi.co/api/v2/version/6/'}},
 {'game_index': 1,
  'version': {'name': 'ruby', 'url': 'https://pokeapi.co/api/v2/version/7/'}},
 {'game_index': 1,
  'version': {'name': 'sapphire',
   'url': 'https://pokeapi.co/api/v2/version/8/'}},
 {'game_index': 1,
  'version': {'name': 'emerald',
   'url': 'https://pokeapi.co/api/v2/version/9/'}},
 {'game_index': 1,
  'version': {'name': 'firered',
   'url': 'https://pokeapi.co/ap

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

In [40]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import threading

BASE_URL = "https://pokeapi.co/api/v2"
MAX_WORKERS = 12

session = requests.Session()

# Thread-safe caches
MOVE_CACHE = {}
MOVE_CACHE_LOCK = threading.Lock()


def log(msg):
    print(f"[INFO] {msg}")


def fetch_json(url):
    r = session.get(url)
    r.raise_for_status()
    return r.json()


def get_german_name(names):
    for n in names:
        if n["language"]["name"] == "de":
            return n["name"]
    return None


# -------------------------------
# Pokémon & Move Localization
# -------------------------------

def get_pokemon_german_name(pid):
    data = fetch_json(f"{BASE_URL}/pokemon-species/{pid}/")
    return get_german_name(data["names"])


def get_move_german_name(move_url):
    with MOVE_CACHE_LOCK:
        if move_url in MOVE_CACHE:
            return MOVE_CACHE[move_url]

    data = fetch_json(move_url)
    name_de = get_german_name(data["names"])

    with MOVE_CACHE_LOCK:
        MOVE_CACHE[move_url] = name_de

    return name_de


# -------------------------------
# Core Data Extraction
# -------------------------------

def parse_learn_method(detail):
    method = detail["move_learn_method"]["name"]

    if method == "level-up":
        return ("level-up", detail["level_learned_at"])
    elif method == "machine":
        return ("machine", None)
    elif method == "tutor":
        return ("tutor", None)
    elif method == "egg":
        return ("egg", None)
    else:
        return (method, None)


def get_pokemon_move_data(pid):
    data = fetch_json(f"{BASE_URL}/pokemon/{pid}/")

    pokemon_moves = defaultdict(lambda: defaultdict(list))
    move_urls = set()

    for move_entry in data["moves"]:
        move_url = move_entry["move"]["url"]
        move_urls.add(move_url)

        for detail in move_entry["version_group_details"]:
            method, level = parse_learn_method(detail)

            if method == "level-up":
                pokemon_moves[move_url][method].append(level)
            else:
                pokemon_moves[move_url][method].append(None)

    return pokemon_moves, move_urls


# -------------------------------
# Main
# -------------------------------

def main():
    pokemon_ids = list(range(1, 1000))

    log("Fetching Pokémon German names...")
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        name_futures = {
            ex.submit(get_pokemon_german_name, pid): pid
            for pid in pokemon_ids
        }
        pokemon_names = {}
        for f in as_completed(name_futures):
            pid = name_futures[f]
            pokemon_names[pid] = f.result()

    log("Fetching Pokémon move learn data...")
    all_move_urls = set()
    pokemon_move_raw = {}

    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futures = {
            ex.submit(get_pokemon_move_data, pid): pid
            for pid in pokemon_ids
        }
        for f in as_completed(futures):
            pid = futures[f]
            move_data, move_urls = f.result()
            pokemon_move_raw[pid] = move_data
            all_move_urls.update(move_urls)
            log(f"Collected moves for Pokémon #{pid}")

    log(f"Fetching German names for {len(all_move_urls)} unique moves...")
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futures = [
            ex.submit(get_move_german_name, url)
            for url in all_move_urls
        ]
        for i, f in enumerate(as_completed(futures), 1):
            f.result()
            if i % 25 == 0 or i == len(futures):
                log(f"Resolved {i}/{len(futures)} move names")

    log("Assembling final result...")

    result = {}

    for pid in pokemon_ids:
        pname = pokemon_names[pid]
        result[pname] = {}

        for move_url, methods in pokemon_move_raw[pid].items():
            move_name = MOVE_CACHE[move_url]
            result[pname][move_name] = dict(methods)

    # -------------------------------
    # Example output preview
    # -------------------------------
    for pokemon, moves in result.items():
        print("\n" + "=" * 60)
        print(pokemon)
        for move, methods in moves.items():
            print(f"  {move}:")
            for method, values in methods.items():
                if method == "level-up":
                    print(f"    - level-up at {sorted(set(values))}")
                else:
                    print(f"    - {method}")

    return result

result = main()


[INFO] Fetching Pokémon German names...
[INFO] Fetching Pokémon move learn data...
[INFO] Collected moves for Pokémon #11
[INFO] Collected moves for Pokémon #8
[INFO] Collected moves for Pokémon #9
[INFO] Collected moves for Pokémon #1
[INFO] Collected moves for Pokémon #13
[INFO] Collected moves for Pokémon #14
[INFO] Collected moves for Pokémon #4
[INFO] Collected moves for Pokémon #7
[INFO] Collected moves for Pokémon #15
[INFO] Collected moves for Pokémon #2
[INFO] Collected moves for Pokémon #6
[INFO] Collected moves for Pokémon #5
[INFO] Collected moves for Pokémon #16
[INFO] Collected moves for Pokémon #21
[INFO] Collected moves for Pokémon #3
[INFO] Collected moves for Pokémon #10
[INFO] Collected moves for Pokémon #19
[INFO] Collected moves for Pokémon #26
[INFO] Collected moves for Pokémon #20
[INFO] Collected moves for Pokémon #28
[INFO] Collected moves for Pokémon #17
[INFO] Collected moves for Pokémon #18
[INFO] Collected moves for Pokémon #24
[INFO] Collected moves for Po

In [25]:
%ls ..\Violet\pokeapi\pokeapi_stats.pkl

 Volume in Laufwerk Z: hat keine Bezeichnung.
 Volumeseriennummer: 0000-0000

 Verzeichnis von Z:\home\wodka\romhacking\Violet\Violet\pokeapi

13.10.2024  15:33           445.434 pokeapi_stats.pkl
               1 Datei(en),        445.434 Bytes
               0 Verzeichnis(se), 972.143.497.216 Bytes frei


In [27]:
with open(r"..\Violet\pokeapi\pokeapi_stats.pkl", "rb") as f:
    import pickle
    legacy = pickle.load(f)

In [31]:
legacy[1]

{'name': {'LANG_GER': 'Bisasam', 'LANG_EN': 'Bulbasaur'},
 'egg_group_0': 'Monster',
 'egg_group_1': 'Pflanze',
 'gender_ratio': 32,
 'capture_rate': 45,
 'base_happiness': 70,
 'egg_cycles': 20,
 'color_and_flip': {'flip': 0, 'color': 'Grün'},
 'shape': 'Quadruped',
 'growth_rate': 'medium-slow',
 'evolutions': {'POKEMON_BISAKNOSP': {'trigger': 'Level_Up',
   'argument': 16,
   'baby_trigger_item': None}},
 'dex_entry': {'LANG_GER': 'Nach der Geburt nimmt es für eine Weile Nährstoffe über den Samen auf seinem Rücken auf.',
  'LANG_EN': 'For some time after its birth, it grows by gaining nourishment from the seed on its back.'},
 'genus': {'LANG_GER': 'Samen', 'LANG_EN': 'Seed'},
 'height': 7,
 'weight': 69,
 'exp_yield': 64,
 'hidden_ability': 'Chlorophyll',
 'ability_0': 'Notdünger',
 'ability_1': None,
 'basestats': {'hp': 45,
  'attack': 49,
  'defense': 49,
  'special-attack': 65,
  'special-defense': 65,
  'speed': 45},
 'ev_yield': {'Padding': 0,
  'hp': 0,
  'attack': 0,
  'def

In [84]:
from copy import deepcopy
updated = deepcopy(legacy)
for l, u in zip(legacy, updated):
    if not l: continue
    name = l["name"]["LANG_GER"]
    if name not in result:
        raise ValueError(f"Missing Pokémon: {name}")
    legacy_moves = set(l["egg_moves"]) | set(m for (m, _) in l["levelup_moves"]) | set(l["accessible_moves"])
    all_new_moves = set(result[name].keys())
    missing_moves = all_new_moves - legacy_moves
    removed_moves = legacy_moves - all_new_moves

    # Assemble a nve levelup move list merging legacy moves and the new moves that are learned by level-up
    # The levelup moves is a list of (move, level) tuples with level being the minimum level for all occurrences
    levelup_moves_new = defaultdict(set)
    for move, methods in result[name].items():
        if "level-up" in methods:
            levelup_moves_new[move].update(l for l in methods["level-up"] if l is not None)

    for move, level in l["levelup_moves"]:
        levelup_moves_new[move].add(level)
        if move not in levelup_moves_new:
            levelup_moves_new[move] = level
    if None in levelup_moves_new:
        del levelup_moves_new[None]
    levelup_moves_list_new = []
    for move, levels in levelup_moves_new.items():
        low_lvls = tuple(lv for lv in levels if lv <= 1)
        high_lvls = tuple(lv for lv in levels if lv > 1)
        if low_lvls:
            levelup_moves_list_new.append((move, min(low_lvls)))
        if high_lvls:
            levelup_moves_list_new.append((move, min(high_lvls)))
    levelup_moves_list_new = sorted(levelup_moves_list_new, key=lambda x: x[1])
    try:
        levelup_moves_new = levelup_moves_list_new
    except Exception as e:
        print(f"Error processing Pokémon {name}: {e}, {levelup_moves_new}")

    # Assemble a new set of egg-moves
    egg_moves_new = set(move for move, methods in result.items() if "egg" in methods)
    egg_moves_new.update(l["egg_moves"])

    accessible_moves_new = set(k for k in result[name].keys() if k) | l["accessible_moves"]
    u["egg_moves"] = sorted(egg_moves_new)
    u["accessible_moves"] = sorted(accessible_moves_new)
    u["levelup_moves"] = list(levelup_moves_new)


    print(f"Pokémon {name} levelup moves updated: {levelup_moves_new}")


    #if missing_moves:
    #    print(f"Pokémon {name} is missing moves: {missing_moves}")
    #if removed_moves:
    #   print(f"Pokémon {name} has removed moves: {removed_moves}")



Pokémon Bisasam levelup moves updated: [('Tackle', 1), ('Heuler', 1), ('Rankenhieb', 3), ('Heuler', 3), ('Wachstum', 6), ('Egelsamen', 7), ('Rasierblatt', 12), ('Giftpuder', 13), ('Schlafpuder', 13), ('Bodycheck', 15), ('Samenbomben', 18), ('Lockduft', 21), ('Risikotackle', 27), ('Synthese', 27), ('Sorgensamen', 30), ('Blattgeißel', 33), ('Solarstrahl', 36)]
Pokémon Bisaknosp levelup moves updated: [('Rankenhieb', 1), ('Tackle', 1), ('Heuler', 1), ('Egelsamen', 1), ('Wachstum', 1), ('Heuler', 3), ('Rankenhieb', 5), ('Egelsamen', 7), ('Rasierblatt', 12), ('Giftpuder', 13), ('Schlafpuder', 13), ('Bodycheck', 15), ('Samenbomben', 20), ('Lockduft', 23), ('Wachstum', 28), ('Risikotackle', 31), ('Synthese', 35), ('Sorgensamen', 36), ('Solarstrahl', 44), ('Blattgeißel', 45)]
Pokémon Bisaflor levelup moves updated: [('Blättertanz', 0), ('Blütenwirbel', 0), ('Rankenhieb', 1), ('Tackle', 1), ('Heuler', 1), ('Egelsamen', 1), ('Wachstum', 1), ('Amnesie', 1), ('Blattgeißel', 1), ('Heuler', 3), ('Ra

In [86]:
with open(r"..\Violet\pokeapi\pokeapi_stats.pkl", "wb") as f:
    import pickle
    legacy = pickle.dump(updated, f)

In [76]:
legacy[3]

{'name': {'LANG_GER': 'Bisaflor', 'LANG_EN': 'Venusaur'},
 'egg_group_0': 'Monster',
 'egg_group_1': 'Pflanze',
 'gender_ratio': 32,
 'capture_rate': 45,
 'base_happiness': 70,
 'egg_cycles': 20,
 'color_and_flip': {'flip': 0, 'color': 'Grün'},
 'shape': 'Quadruped',
 'growth_rate': 'medium-slow',
 'evolutions': {},
 'dex_entry': {'LANG_GER': 'Nach einem Regentag riecht die Blume auf seinem Rücken intensiver. Das Aroma zieht andere Pokémon an.',
  'LANG_EN': 'After a rainy day, the flower on its back smells stronger. The scent attracts other Pokémon.'},
 'genus': {'LANG_GER': 'Samen', 'LANG_EN': 'Seed'},
 'height': 20,
 'weight': 1000,
 'exp_yield': 236,
 'hidden_ability': 'Chlorophyll',
 'ability_0': 'Notdünger',
 'ability_1': None,
 'basestats': {'hp': 80,
  'attack': 82,
  'defense': 83,
  'special-attack': 100,
  'special-defense': 100,
  'speed': 80},
 'ev_yield': {'Padding': 0,
  'hp': 0,
  'attack': 0,
  'defense': 0,
  'special-attack': 2,
  'special-defense': 1,
  'speed': 0},

In [87]:
import json
x = json.loads("""
{
	"label": "pokemon_accessible_moves",
	"type": "accessible_moves",
	"data": [
		[],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_KLINGENSTURM",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_WUTANFALL",
			"ATTACK_VERWURZLER",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_BLAETTERTANZ",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_WUTANFALL",
			"ATTACK_VERWURZLER",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_FLUEGELSCHLAG",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_METEOROLOGE",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_GEDULD",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_PRUEGLER",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BAUCHTROMMEL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_KNIRSCHER",
			"ATTACK_METEOROLOGE",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_GEDULD",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_BISS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HITZEWELLE",
			"ATTACK_DRACHENTANZ",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_FEUERFEGER",
			"ATTACK_GEOFISSUR",
			"ATTACK_PRUEGLER",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BAUCHTROMMEL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_BISS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_DRACHENTANZ",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_KONFUSION",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TAUCHER",
			"ATTACK_EISHIEB",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_GAEHNER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MOGELHIEB",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FONTRAENEN",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TAUCHER",
			"ATTACK_WEISSNEBEL",
			"ATTACK_EISHIEB",
			"ATTACK_WHIRLPOOL",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_SPIEGELCAPE",
			"ATTACK_EINIGLER",
			"ATTACK_WASSERDUESE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_GAEHNER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_KNIRSCHER",
			"ATTACK_METEOROLOGE",
			"ATTACK_GEDULD",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FONTRAENEN",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TAUCHER",
			"ATTACK_WEISSNEBEL",
			"ATTACK_EISHIEB",
			"ATTACK_WHIRLPOOL",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_SPIEGELCAPE",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STURZFLUG",
			"ATTACK_STERNSCHAUER",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_SCHNARCHER",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_HITZEWELLE",
			"ATTACK_RISIKOTACKLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STURZFLUG",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HITZEWELLE"
		],
		[],
		[
			"ATTACK_FADENSCHUSS",
			"ATTACK_EISENABWEHR",
			"ATTACK_KAEFERBISS"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_SILBERHAUCH",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_LOCKDUFT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FELSWURF",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_NACHTNEBEL",
			"ATTACK_FELSWURF",
			"ATTACK_PSYSTRAHL",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_KNUDDLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_KNUDDLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUSSETZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_GROLL",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUSSETZER",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_BODYCHECK",
			"ATTACK_LEIDTEILER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KNUDDLER",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_VOLTTACKLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DIEBESKUSS",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_EINIGLER",
			"ATTACK_LADEVORGANG",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KNUDDLER",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DIEBESKUSS",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_EINIGLER",
			"ATTACK_LADEVORGANG",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GEOFISSUR",
			"ATTACK_BLUTSAUGER",
			"ATTACK_NOTSITUATION",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_FUSSKICK",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GEOFISSUR",
			"ATTACK_BLUTSAUGER",
			"ATTACK_NOTSITUATION",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_METALLKLAUE",
			"ATTACK_NADELRAKETE"
		],
		[
			"ATTACK_GEDULD",
			"ATTACK_KOPFNUSS",
			"ATTACK_SCHNARCHER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_EINIGLER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_PRUEGLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KOPFNUSS",
			"ATTACK_HORNBOHRER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GEOFISSUR",
			"ATTACK_PRUEGLER",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_HORNBOHRER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_FUCHTLER",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_PRUEGLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GEOFISSUR",
			"ATTACK_PRUEGLER",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_AMNESIE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_HORNBOHRER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_VITALGLOCKE",
			"ATTACK_STAFETTE",
			"ATTACK_DIEBESKUSS",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_VITALGLOCKE",
			"ATTACK_STAFETTE",
			"ATTACK_DIEBESKUSS",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_AGILITAET",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEMENTO_MORI",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_JAULER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_NACHTNEBEL",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_AGILITAET",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEMENTO_MORI",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_AMNESIE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_DIEBESKUSS",
			"ATTACK_STAFETTE",
			"ATTACK_KONTER",
			"ATTACK_KREIDESCHREI",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCANNER",
			"ATTACK_PSYWELLE",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_AQUAKNARRE",
			"ATTACK_TELEPORT",
			"ATTACK_METRONOM",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_AMNESIE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_DIEBESKUSS",
			"ATTACK_STAFETTE",
			"ATTACK_KONTER",
			"ATTACK_KREIDESCHREI",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCANNER",
			"ATTACK_PSYWELLE",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_AQUAKNARRE",
			"ATTACK_TELEPORT",
			"ATTACK_METRONOM",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KLINGENSTURM",
			"ATTACK_WINDHOSE",
			"ATTACK_AGILITAET",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_STURZFLUG",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KLINGENSTURM",
			"ATTACK_WINDHOSE",
			"ATTACK_AGILITAET",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_EGELSAMEN",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_VERWURZLER",
			"ATTACK_GEDULD",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_TAUMELTANZ",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_EGELSAMEN",
			"ATTACK_RASEREI",
			"ATTACK_SYNTHESE",
			"ATTACK_SPASSKANONE",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RASIERBLATT",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_VERWURZLER",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_TAUMELTANZ",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_EGELSAMEN",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SYNTHESE",
			"ATTACK_SPASSKANONE",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_TRUGTRAENE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_METEOROLOGE",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_METEOROLOGE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_BODYSLAM",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_SPRUNGFEDER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_GIFTSTACHEL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LADEVORGANG",
			"ATTACK_NADELRAKETE"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_PRUEGLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_RASEREI"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_STACHLER",
			"ATTACK_SAMENBOMBEN"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACHENWUT",
			"ATTACK_CHARME",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_FEUERODEM",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACHENWUT",
			"ATTACK_CHARME",
			"ATTACK_WUTANFALL",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_FEUERODEM",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_BEZIRZER",
			"ATTACK_NAHKAMPF",
			"ATTACK_TELEPORT",
			"ATTACK_HITZEWELLE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_NAHKAMPF",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KREIDESCHREI"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_NAHKAMPF",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KREIDESCHREI"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_NAHKAMPF",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KREIDESCHREI"
		],
		[
			"ATTACK_KONFUSION",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_MEGAHIEB",
			"ATTACK_PSYWELLE",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_MEGAHIEB",
			"ATTACK_PSYWELLE",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_BEGRENZER",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_MEGAHIEB",
			"ATTACK_PSYWELLE",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_FELSWURF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_STERNSCHAUER",
			"ATTACK_BLUTSAUGER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_VERWURZLER",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KAEFERBISS",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SYNTHESE",
			"ATTACK_SPASSKANONE",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_VERWURZLER",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KAEFERBISS",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SYNTHESE",
			"ATTACK_SPASSKANONE",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_RASEREI"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_AURORASTRAHL",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_TURBODREHER",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_SPASSKANONE"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_HAMMERARM",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_HAMMERARM",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KNUDDLER",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_CHARME",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KNUDDLER",
			"ATTACK_FUSSKICK",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_RASEREI",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_TRIPLETTE",
			"ATTACK_BEGRENZER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TAUCHER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_TELEPORT"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_BEGRENZER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_TAUCHER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_EISHIEB",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR",
			"ATTACK_TELEPORT",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_EISENABWEHR",
			"ATTACK_TELEPORT",
			"ATTACK_LADEVORGANG"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_EISENABWEHR",
			"ATTACK_TELEPORT",
			"ATTACK_LADEVORGANG"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_WIRBELWIND",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KLINGENSTURM",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_WINDHOSE",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_NAHKAMPF",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_WIRBELWIND",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_FUSSKICK",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_KOPFNUSS",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_WIRBELWIND",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STURZFLUG",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_GEDULD",
			"ATTACK_FUSSKICK",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_KOPFNUSS",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_VIELENDER",
			"ATTACK_SCHNABEL",
			"ATTACK_WHIRLPOOL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_HYDROPUMPE",
			"ATTACK_DUNKELNEBEL"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_CHARME",
			"ATTACK_HORTER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_AUSSETZER",
			"ATTACK_VIELENDER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_VERZEHRER",
			"ATTACK_EISSPEER",
			"ATTACK_ENTFESSLER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_ABGESANG",
			"ATTACK_SCHLECKER",
			"ATTACK_RASEREI",
			"ATTACK_HYDROPUMPE",
			"ATTACK_DUNKELNEBEL"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_METRONOM",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_BEGRENZER",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_METRONOM",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_STACHLER",
			"ATTACK_TRIPLETTE",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_TELEPORT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_TRIPLETTE",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KREIDESCHREI",
			"ATTACK_RASEREI",
			"ATTACK_TELEPORT",
			"ATTACK_NADELRAKETE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGASAUGER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_BEGRENZER",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGASAUGER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_BEGRENZER",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_PSYWELLE",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_MEGASAUGER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_BEGRENZER",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_PSYWELLE",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_DRACHENTANZ"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_GEDULD",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_FUSSKICK",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_ZUGABE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_TELEPORT",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_FUSSKICK",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_DIEBESKUSS",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_TELEPORT",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_NACHTHIEB",
			"ATTACK_SCHLITZER",
			"ATTACK_HAMMERARM",
			"ATTACK_TAUCHER",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_METALLSOUND",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_AGILITAET",
			"ATTACK_RASEREI",
			"ATTACK_TELEPORT"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METALLSOUND",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_AGILITAET",
			"ATTACK_RASEREI",
			"ATTACK_TELEPORT"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_BEGRENZER",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_EIERBOMBE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_TELEPORT"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FUSSKICK",
			"ATTACK_BEGRENZER",
			"ATTACK_SEHER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_TELEPORT",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EISENABWEHR",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EISENABWEHR",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TURBODREHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_TEMPOHIEB",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_PATRONENHIEB",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_TURBODREHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_TURMKICK",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_STAFETTE",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_NACHTMAHR",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_AQUAKNARRE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_VERFOLGUNG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_FUCHTLER"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_GEDULD",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_AQUAKNARRE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_PSYWELLE",
			"ATTACK_AQUAKNARRE",
			"ATTACK_TELEPORT",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_PSYSTRAHL",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_METALLKLAUE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_PRUEGLER",
			"ATTACK_GROLL",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_AQUAKNARRE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_BEZIRZER",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_WUTANFALL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AURORASTRAHL",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_PLATSCHER"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_BEGRENZER",
			"ATTACK_SCHNARCHER",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_TAUCHER",
			"ATTACK_PSYCHO_PLUS"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_BEGRENZER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCHNARCHER",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_TAUCHER",
			"ATTACK_PSYCHO_PLUS"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_SUPERSCHALL",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_TRIPLETTE",
			"ATTACK_AURORASTRAHL",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDHOSE",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_BARRIERE",
			"ATTACK_TELEPORT"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_TRIPLETTE",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDHOSE",
			"ATTACK_AGILITAET",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_TELEPORT"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_CHARME",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_NACHTNEBEL",
			"ATTACK_SEHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_MIMIKRY",
			"ATTACK_DIEBESKUSS",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KAEFERBISS",
			"ATTACK_WINDSCHNITT",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_STERNSCHAUER",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FUSSKICK",
			"ATTACK_NACHTNEBEL",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_AGILITAET",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_HAMMERARM",
			"ATTACK_METALLSOUND",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_PSYWELLE",
			"ATTACK_SPOTLIGHT",
			"ATTACK_BEZIRZER",
			"ATTACK_TELEPORT",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB",
			"ATTACK_KREUZHIEB"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BAUCHTROMMEL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_TEMPOHIEB",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_KREIDESCHREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_MEGAHIEB",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_PSYWELLE",
			"ATTACK_SPOTLIGHT",
			"ATTACK_BEZIRZER",
			"ATTACK_TELEPORT",
			"ATTACK_HITZEWELLE",
			"ATTACK_METRONOM",
			"ATTACK_KREUZHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_GEDULD",
			"ATTACK_FADENSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_GEOFISSUR",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_VIELENDER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_HORNBOHRER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_NAHKAMPF"
		],
		[
			"ATTACK_SPRUNGFEDER",
			"ATTACK_HYDROPUMPE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_FEUERODEM",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_AQUAKNARRE"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACHENWUT",
			"ATTACK_BLITZKANONE",
			"ATTACK_CHARME",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_AURORASTRAHL",
			"ATTACK_FEUERODEM",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_TAUCHER",
			"ATTACK_VIELENDER",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISSPEER",
			"ATTACK_VITALGLOCKE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_SCHALLWELLE"
		],
		[],
		[
			"ATTACK_ZAHLTAG",
			"ATTACK_SCHNARCHER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_SCHALLWELLE",
			"ATTACK_METALLSOUND",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_LADEVORGANG"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_PSYWELLE",
			"ATTACK_TELEPORT",
			"ATTACK_LADEVORGANG"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_STACHLER",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_HORNBOHRER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_EISENABWEHR",
			"ATTACK_NADELRAKETE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_FELSWURF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_FUSSKICK",
			"ATTACK_FELSWURF",
			"ATTACK_TAUCHER",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KLINGENSTURM",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_AQUAKNARRE",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACHENWUT",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_FELSWURF",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_KLINGENSTURM",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE",
			"ATTACK_DRACHENTANZ"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_ZAHLTAG",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_PSYWELLE",
			"ATTACK_AQUAKNARRE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_WIRBELWIND",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STURZFLUG",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_SCHNABEL",
			"ATTACK_KOPFNUSS",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KLINGENSTURM",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_AQUAKNARRE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_WIRBELWIND",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STURZFLUG",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_METALLSOUND",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KLINGENSTURM",
			"ATTACK_WINDSCHNITT",
			"ATTACK_STAFETTE",
			"ATTACK_WINDHOSE",
			"ATTACK_RASEREI",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_WIRBELWIND",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STURZFLUG",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_SCHNABEL",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KLINGENSTURM",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_AQUAKNARRE",
			"ATTACK_HYDROPUMPE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_FEUERODEM",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_HORNBOHRER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_AQUAKNARRE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_DUNKELNEBEL"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_FEUERODEM",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_HORNBOHRER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KLINGENSTURM",
			"ATTACK_WINDSCHNITT",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_AQUAKNARRE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_HITZEWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_BLITZKANONE",
			"ATTACK_ZAHLTAG",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_BEGRENZER",
			"ATTACK_FUSSKICK",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TAUCHER",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_AQUAKNARRE",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_METALLKLAUE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_STURZFLUG",
			"ATTACK_FEUERODEM",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISSPEER",
			"ATTACK_VITALGLOCKE",
			"ATTACK_HORNBOHRER",
			"ATTACK_BODYCHECK",
			"ATTACK_KAEFERBISS",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_RASEREI",
			"ATTACK_KREIDESCHREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_HYDROPUMPE",
			"ATTACK_BLAETTERSTURM",
			"ATTACK_FEUERFEGER",
			"ATTACK_SUPERZAHN",
			"ATTACK_GEOFISSUR",
			"ATTACK_WIRBELWIND",
			"ATTACK_RECHTE_HAND",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MOGELHIEB",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_MEGASAUGER",
			"ATTACK_KNIRSCHER",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METALLSOUND",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_LAUBKLINGE",
			"ATTACK_EIERBOMBE",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_KLINGENSTURM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_AGILITAET",
			"ATTACK_AQUAKNARRE",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_VERGELTUNG",
			"ATTACK_PRUEGLER",
			"ATTACK_UEBERROLLER",
			"ATTACK_DRACHENWUT",
			"ATTACK_GEDULD",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FINTE",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_SILBERHAUCH",
			"ATTACK_VIELENDER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_WEICHEI",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_BEZIRZER",
			"ATTACK_GIFTSCHWEIF",
			"ATTACK_NAHKAMPF",
			"ATTACK_SANDGRAB",
			"ATTACK_HYPNOSE",
			"ATTACK_LADEVORGANG",
			"ATTACK_BLUTSAUGER",
			"ATTACK_NOTSITUATION",
			"ATTACK_KOSMIK_KRAFT",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_GRIMASSE",
			"ATTACK_ZAHLTAG",
			"ATTACK_CHARME",
			"ATTACK_NACHTMAHR",
			"ATTACK_DRACO_METEOR",
			"ATTACK_WUTANFALL",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_STACHLER",
			"ATTACK_TRIPLETTE",
			"ATTACK_FUSSKICK",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_FADENSCHUSS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_FELSWURF",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_NADELRAKETE",
			"ATTACK_TAUCHER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_SEHER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_KOPFNUSS",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_SYNTHESE",
			"ATTACK_EISENABWEHR",
			"ATTACK_TELEPORT",
			"ATTACK_LOCKDUFT",
			"ATTACK_DRACHENTANZ",
			"ATTACK_SCHALLWELLE",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_TRUGTRAENE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_VERWURZLER",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_RANKENHIEB",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KONTER",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_TRUGTRAENE",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_VERWURZLER",
			"ATTACK_METEOROLOGE",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_RANKENHIEB",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KONTER",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_KNUDDLER",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_SCANNER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_JAULER",
			"ATTACK_KNUDDLER",
			"ATTACK_ZORNKLINGE",
			"ATTACK_DOPPELKICK",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_SONDERSENSOR",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_BEZIRZER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_JAULER",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KNUDDLER",
			"ATTACK_FUSSKICK",
			"ATTACK_NACHTNEBEL",
			"ATTACK_DOPPELKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_SONDERSENSOR",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_BEZIRZER",
			"ATTACK_HITZEWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KLINGENSTURM",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_SCHMEICHLER",
			"ATTACK_TAUCHER",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_DRACHENTANZ",
			"ATTACK_METALLKLAUE",
			"ATTACK_WASSERDUESE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_SCHMEICHLER",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TAUCHER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_DRACHENTANZ",
			"ATTACK_METALLKLAUE",
			"ATTACK_WASSERDUESE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_FEUERWIRBEL",
			"ATTACK_SCHNARCHER",
			"ATTACK_BODYSLAM",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDSCHNITT"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_METEOROLOGE",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_STURZFLUG",
			"ATTACK_AUFRUHR",
			"ATTACK_FUSSKICK",
			"ATTACK_HORRORBLICK",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_NAHKAMPF",
			"ATTACK_MEGAHIEB",
			"ATTACK_KREUZHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FADENSCHUSS",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_MEGAHIEB",
			"ATTACK_LOCKDUFT",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FADENSCHUSS",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_KAEFERBISS",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_MEGAHIEB",
			"ATTACK_LOCKDUFT",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KAEFERBISS"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_NACHTHIEB",
			"ATTACK_AUSSETZER",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_VIELENDER",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KNIRSCHER",
			"ATTACK_STURZFLUG",
			"ATTACK_AUFRUHR",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDHOSE",
			"ATTACK_AGILITAET",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_VITALGLOCKE",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY"
		],
		[
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_WEISSNEBEL",
			"ATTACK_PSYSTRAHL",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_AMNESIE",
			"ATTACK_WHIRLPOOL",
			"ATTACK_VITALGLOCKE",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_AGILITAET",
			"ATTACK_KREIDESCHREI"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_EISENABWEHR",
			"ATTACK_LADEVORGANG"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_BEGRENZER",
			"ATTACK_SILBERHAUCH",
			"ATTACK_SEHER",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_BEZIRZER",
			"ATTACK_HITZEWELLE",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_VITALGLOCKE",
			"ATTACK_STAFETTE",
			"ATTACK_DIEBESKUSS",
			"ATTACK_BEZIRZER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_TRIPLETTE",
			"ATTACK_KOPFNUSS",
			"ATTACK_WEICHEI",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_TRIPLETTE",
			"ATTACK_BEGRENZER",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_SILBERHAUCH",
			"ATTACK_SEHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WEICHEI",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_KOSMIK_KRAFT",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_SILBERHAUCH",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_STAFETTE",
			"ATTACK_WINDHOSE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_KOSMIK_KRAFT",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_SILBERHAUCH",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_STAFETTE",
			"ATTACK_WINDHOSE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_MIMIKRY",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_VITALGLOCKE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_AGILITAET",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_VITALGLOCKE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_AGILITAET",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RASIERBLATT",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_VERWURZLER",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_ZUGABE",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_TAUMELTANZ",
			"ATTACK_MIMIKRY",
			"ATTACK_EGELSAMEN",
			"ATTACK_STAFETTE",
			"ATTACK_SYNTHESE",
			"ATTACK_SPASSKANONE"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_GESANG",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_DIEBESKUSS",
			"ATTACK_SPASSKANONE",
			"ATTACK_BEZIRZER",
			"ATTACK_GESICHTE",
			"ATTACK_METRONOM",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_SEHER",
			"ATTACK_TAUCHER",
			"ATTACK_AMNESIE",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_BEZIRZER",
			"ATTACK_PSYCHO_PLUS"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FLAMMENBLITZ"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_SAEUREPANZER",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_SAEUREPANZER",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MEGAHIEB",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_SYNTHESE",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_METEOROLOGE",
			"ATTACK_FUSSKICK",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_SYNTHESE",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_PRUEGLER",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SLAM",
			"ATTACK_SCHNARCHER",
			"ATTACK_MOGELHIEB",
			"ATTACK_AUFRUHR",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EINIGLER",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_METEOROLOGE",
			"ATTACK_BEGRENZER",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_KNIRSCHER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDHOSE",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE",
			"ATTACK_RUCKZUCKHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_BEGRENZER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_TAUCHER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDHOSE",
			"ATTACK_HITZEWELLE"
		],
		[],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_STACHLER",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_LEIDTEILER",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_FLUEGELSCHLAG",
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KNIRSCHER",
			"ATTACK_STACHLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_KAEFERBISS",
			"ATTACK_STAFETTE",
			"ATTACK_KONTER",
			"ATTACK_AGILITAET",
			"ATTACK_GIFTSCHWEIF",
			"ATTACK_METALLKLAUE",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_FADENSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_LOCKDUFT",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_METALLSOUND",
			"ATTACK_FADENSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_LEIDTEILER",
			"ATTACK_LOCKDUFT",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_HAMMERARM",
			"ATTACK_METALLSOUND",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_SPOTLIGHT",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB",
			"ATTACK_KREUZHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_KNIRSCHER",
			"ATTACK_STACHLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KAEFERBISS",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FELSWURF",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_EISENABWEHR",
			"ATTACK_DRACHENTANZ",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_BLITZKANONE",
			"ATTACK_SILBERBLICK",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_GESCHENK",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_NAHKAMPF",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_BISS",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISSPEER",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KAEFERBISS",
			"ATTACK_WINDSCHNITT",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_STAFETTE",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_KONTER",
			"ATTACK_KREIDESCHREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_STACHLER",
			"ATTACK_FUSSKICK",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KAEFERBISS",
			"ATTACK_SCANNER",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_AUSSETZER",
			"ATTACK_SILBERHAUCH",
			"ATTACK_PSYSTRAHL",
			"ATTACK_BODYSLAM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_NAHKAMPF",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_METALLKLAUE",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEIDTEILER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_EISENABWEHR",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_RAUCHWOLKE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_HORTER",
			"ATTACK_SAEUREPANZER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEMENTO_MORI",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_EISENABWEHR",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_BISS",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISSPEER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_MEGAKICK",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HYDROPUMPE",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_RISIKOTACKLE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_TAUCHER",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TAUCHER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_KREIDESCHREI",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_STURZFLUG",
			"ATTACK_METEOROLOGE",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_EISSPEER",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_STAFETTE",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_AGILITAET",
			"ATTACK_SCANNER",
			"ATTACK_MEMENTO_MORI",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_AMNESIE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HYDROPUMPE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ZORNKLINGE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_WINDHOSE",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_HITZEWELLE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCANNER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_HITZEWELLE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUSSETZER",
			"ATTACK_AURORASTRAHL",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_PLATSCHER"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_MIMIKRY",
			"ATTACK_AQUAKNARRE",
			"ATTACK_SANDGRAB",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_KNUDDLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_EISENABWEHR",
			"ATTACK_SANDGRAB",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEIDTEILER",
			"ATTACK_LADEVORGANG"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_METALLSOUND",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_SANDGRAB",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_SEHER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_VITALGLOCKE",
			"ATTACK_DIEBESKUSS",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_FUSSKICK",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_FUSSKICK",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_TURMKICK",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_STAFETTE",
			"ATTACK_WINDHOSE",
			"ATTACK_BODYSLAM",
			"ATTACK_TEMPOHIEB",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_PATRONENHIEB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_AMNESIE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_SPASSKANONE",
			"ATTACK_HITZEWELLE",
			"ATTACK_DRACHENTANZ"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_METALLSOUND",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_SPOTLIGHT",
			"ATTACK_BEZIRZER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_SPOTLIGHT",
			"ATTACK_BEZIRZER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LOCKDUFT",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_GESCHENK",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_AGILITAET",
			"ATTACK_SCANNER",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET",
			"ATTACK_SCANNER"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_SCANNER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_DRACHENTANZ",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FUSSKICK",
			"ATTACK_FEUERODEM",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_HYDROPUMPE",
			"ATTACK_DRACHENTANZ",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_STURZFLUG",
			"ATTACK_BEGRENZER",
			"ATTACK_FEUERODEM",
			"ATTACK_TAUCHER",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_SCANNER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_FEUERODEM",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_SCANNER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HITZEWELLE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_BEGRENZER",
			"ATTACK_LAUBKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_SILBERHAUCH",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_SCANNER",
			"ATTACK_SYNTHESE",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_LOCKDUFT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_NAHKAMPF",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KREIDESCHREI"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GEOFISSUR",
			"ATTACK_BLUTSAUGER",
			"ATTACK_NOTSITUATION",
			"ATTACK_UEBERROLLER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_BEZIRZER",
			"ATTACK_METALLKLAUE",
			"ATTACK_NADELRAKETE"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_BLITZKANONE",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_TRIPLETTE",
			"ATTACK_BEGRENZER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_TAUCHER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_EISHIEB",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_PSYWELLE",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR",
			"ATTACK_TELEPORT",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_BLUTSAUGER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_PSYSTRAHL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KAEFERBISS",
			"ATTACK_WINDHOSE",
			"ATTACK_HYDROPUMPE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_VITALGLOCKE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_AGILITAET",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_SILBERHAUCH",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_LOCKDUFT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_STURZFLUG",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_AGILITAET",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_HITZEWELLE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_UEBERROLLER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_HAMMERARM",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BAUCHTROMMEL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_STAFETTE",
			"ATTACK_AGILITAET",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_GEGENSCHLAG"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_PSYSTRAHL",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_METALLKLAUE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_SILBERHAUCH",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TAUCHER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_KREIDESCHREI",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_UEBERROLLER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SCANNER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_NACHTHIEB",
			"ATTACK_AUSSETZER",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_VIELENDER",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STACHLER",
			"ATTACK_EISSPEER",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_CHARME",
			"ATTACK_HORTER",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_AUSSETZER",
			"ATTACK_VIELENDER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_VERZEHRER",
			"ATTACK_EISSPEER",
			"ATTACK_ENTFESSLER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_ABGESANG",
			"ATTACK_SCHLECKER",
			"ATTACK_RASEREI",
			"ATTACK_HYDROPUMPE",
			"ATTACK_DUNKELNEBEL"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_NACHTMAHR",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_PSYSTRAHL",
			"ATTACK_ZUGABE",
			"ATTACK_MIMIKRY",
			"ATTACK_ABGESANG",
			"ATTACK_STAFETTE",
			"ATTACK_DIEBESKUSS",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_GEDULD",
			"ATTACK_AUFRUHR",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_EGELSAMEN",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_WACHSTUM",
			"ATTACK_SYNTHESE",
			"ATTACK_ABSORBER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_AUFRUHR",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_EGELSAMEN",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_WACHSTUM",
			"ATTACK_SYNTHESE",
			"ATTACK_ABSORBER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGASAUGER",
			"ATTACK_AUFRUHR",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_EGELSAMEN",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_SCANNER",
			"ATTACK_WACHSTUM",
			"ATTACK_SYNTHESE",
			"ATTACK_ABSORBER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_SCHNARCHER",
			"ATTACK_SILBERBLICK",
			"ATTACK_WUTANFALL",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_SCHLITZER",
			"ATTACK_RUTENSCHLAG",
			"ATTACK_FEUERODEM",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_RUCKZUCKHIEB",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_METALLKLAUE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_SCHNARCHER",
			"ATTACK_SILBERBLICK",
			"ATTACK_WUTANFALL",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_SCHLITZER",
			"ATTACK_RUTENSCHLAG",
			"ATTACK_FEUERODEM",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_RUCKZUCKHIEB",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_METALLKLAUE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_SCHNARCHER",
			"ATTACK_SILBERBLICK",
			"ATTACK_WUTANFALL",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_AUFRUHR",
			"ATTACK_SCHLITZER",
			"ATTACK_RUTENSCHLAG",
			"ATTACK_FEUERODEM",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_KONTER",
			"ATTACK_RASEREI",
			"ATTACK_LEIDTEILER",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_RUCKZUCKHIEB",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_METALLKLAUE"
		],
		[
			"ATTACK_NASSSCHWEIF",
			"ATTACK_SCHLITZER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_NOTSITUATION",
			"ATTACK_HORNATTACKE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACHENWUT",
			"ATTACK_BISS",
			"ATTACK_BODYSLAM",
			"ATTACK_WUTANFALL",
			"ATTACK_TAUCHER",
			"ATTACK_RASEREI",
			"ATTACK_WEISSNEBEL"
		],
		[
			"ATTACK_SCHLITZER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_NOTSITUATION",
			"ATTACK_HORNATTACKE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACHENWUT",
			"ATTACK_BISS",
			"ATTACK_BODYSLAM",
			"ATTACK_WUTANFALL",
			"ATTACK_TAUCHER",
			"ATTACK_RASEREI",
			"ATTACK_WEISSNEBEL"
		],
		[
			"ATTACK_SCHLITZER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_NOTSITUATION",
			"ATTACK_HORNATTACKE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACHENWUT",
			"ATTACK_BISS",
			"ATTACK_BODYSLAM",
			"ATTACK_WUTANFALL",
			"ATTACK_TAUCHER",
			"ATTACK_RASEREI",
			"ATTACK_WEISSNEBEL"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_ERSTAUNER",
			"ATTACK_GIFTZAHN",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_BEZIRZER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_WHIRLPOOL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_TURBOTEMPO",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_SCHNARCHER"
		],
		[
			"ATTACK_FADENSCHUSS",
			"ATTACK_KAEFERBISS"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDHOSE"
		],
		[
			"ATTACK_FADENSCHUSS",
			"ATTACK_KAEFERBISS"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE"
		],
		[
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RASIERBLATT",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_METEOROLOGE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_TAUCHER",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SYNTHESE",
			"ATTACK_SPASSKANONE",
			"ATTACK_METRONOM",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_LOCKDUFT",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_STERNSCHAUER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RASIERBLATT",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TAUCHER",
			"ATTACK_AMNESIE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SYNTHESE",
			"ATTACK_SPASSKANONE",
			"ATTACK_METRONOM",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_LOCKDUFT",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_NACHTHIEB",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_PRUEGLER",
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_NACHTHIEB",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_EGELSAMEN",
			"ATTACK_BODYSLAM",
			"ATTACK_RUCKZUCKHIEB",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_PRUEGLER",
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MEGAKICK",
			"ATTACK_NACHTHIEB",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_PFUND",
			"ATTACK_EINIGLER",
			"ATTACK_BEGRENZER",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_SILBERHAUCH",
			"ATTACK_AMNESIE",
			"ATTACK_KOPFNUSS",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_EGELSAMEN",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_WINDHOSE",
			"ATTACK_KREIDESCHREI",
			"ATTACK_BODYSLAM",
			"ATTACK_HITZEWELLE",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_RUCKZUCKHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_SCHNARCHER",
			"ATTACK_FADENSCHUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ZORNKLINGE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_AUFRUHR",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_FADENSCHUSS",
			"ATTACK_SILBERHAUCH",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_FADENSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_MIMIKRY",
			"ATTACK_KAEFERBISS",
			"ATTACK_AGILITAET"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_KONTER",
			"ATTACK_WINDHOSE",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_KONTER",
			"ATTACK_WINDHOSE",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SYNTHESE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ZAUBERBLATT"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_SYNTHESE",
			"ATTACK_NAHKAMPF",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STURZFLUG",
			"ATTACK_AUFRUHR",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_HYDROPUMPE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STURZFLUG",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_WHIRLPOOL",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_BLUTSAUGER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_KAEFERGEBRUMM",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_BLUTSAUGER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_PSYSTRAHL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KAEFERBISS",
			"ATTACK_WINDHOSE",
			"ATTACK_HYDROPUMPE"
		],
		[
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_FINALE",
			"ATTACK_MIMIKRY",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_METEOROLOGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_SEHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_EISENABWEHR",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BAUCHTROMMEL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_TEMPOHIEB",
			"ATTACK_KREIDESCHREI",
			"ATTACK_MEGAHIEB",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_SPOTLIGHT",
			"ATTACK_BEZIRZER",
			"ATTACK_HITZEWELLE",
			"ATTACK_METRONOM",
			"ATTACK_SCHALLWELLE",
			"ATTACK_KREUZHIEB"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEMENTO_MORI",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_WUTANFALL",
			"ATTACK_RISIKOTACKLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_WUTANFALL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_STACHLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_HYDROPUMPE",
			"ATTACK_DRACHENTANZ",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_ZAHLTAG",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR",
			"ATTACK_METALLKLAUE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_SCHLITZER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KONTER",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_GROLL",
			"ATTACK_GRIMASSE",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_SCHLITZER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TAUCHER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_HYDROPUMPE",
			"ATTACK_NAHKAMPF",
			"ATTACK_EISENABWEHR",
			"ATTACK_DRACHENTANZ",
			"ATTACK_METALLKLAUE",
			"ATTACK_WASSERDUESE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_LEHMBRUEHE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_BEGRENZER",
			"ATTACK_FEUERODEM",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_TAUCHER",
			"ATTACK_WEISSNEBEL",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_SPASSKANONE",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_SPIEGELCAPE",
			"ATTACK_DRACHENTANZ",
			"ATTACK_HYPNOSE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_TAUCHER",
			"ATTACK_ZORNKLINGE",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TAUCHER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_HYDROPUMPE",
			"ATTACK_NAHKAMPF"
		],
		[
			"ATTACK_KOPFNUSS",
			"ATTACK_SCHNARCHER",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_RISIKOTACKLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_WINDSTOSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SILBERHAUCH",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_KAEFERBISS",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_HITZEWELLE",
			"ATTACK_RUCKZUCKHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_WINDSTOSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SILBERHAUCH",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_DRESCHFLEGEL",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_KAEFERBISS",
			"ATTACK_WINDSCHNITT",
			"ATTACK_WINDHOSE",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HITZEWELLE",
			"ATTACK_RUCKZUCKHIEB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_JAULER",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_FEUERFEGER",
			"ATTACK_VERGELTUNG",
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ENERGIEFOKUS",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_FEUERWIRBEL"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_CHARME",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_KOPFNUSS",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_HYDROPUMPE"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_SYNTHESE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_BLAETTERSTURM",
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_SYNTHESE",
			"ATTACK_ZAUBERBLATT",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_EISSPEER",
			"ATTACK_SCHNARCHER",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_RISIKOTACKLE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_TRUGTRAENE",
			"ATTACK_GROLL",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_STACHLER",
			"ATTACK_EISSPEER",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FELSWURF",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_LEIDTEILER",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FELSWURF",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_LEIDTEILER",
			"ATTACK_EISENABWEHR",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_SUPERSCHALL",
			"ATTACK_BAUCHTROMMEL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_AUFRUHR",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_GESCHENK",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_ABGESANG",
			"ATTACK_DIEBESKUSS",
			"ATTACK_BEZIRZER",
			"ATTACK_EINIGLER",
			"ATTACK_WASSERDUESE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_AUFRUHR",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_SEHER",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_AMNESIE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_METRONOM",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_BLITZKANONE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_KOPFNUSS",
			"ATTACK_VOLTTACKLE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_SCANNER",
			"ATTACK_SPOTLIGHT",
			"ATTACK_BEZIRZER",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_LEIDTEILER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_MEGAHIEB",
			"ATTACK_LEIDTEILER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_NAHKAMPF",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_BEGRENZER",
			"ATTACK_NACHTNEBEL",
			"ATTACK_KOPFNUSS",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_LEIDTEILER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_NAHKAMPF",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_STURZFLUG",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_HIMMELSFEGER",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_STURZFLUG",
			"ATTACK_METEOROLOGE",
			"ATTACK_AUFRUHR",
			"ATTACK_KNUDDLER",
			"ATTACK_DAUNENREIGEN",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_AGILITAET",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_HITZEWELLE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_VITALGLOCKE",
			"ATTACK_STAFETTE",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM"
		],
		[
			"ATTACK_VERGELTUNG",
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_BEGRENZER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEMENTO_MORI",
			"ATTACK_DUNKELNEBEL",
			"ATTACK_METRONOM",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_BODYCHECK",
			"ATTACK_LEIDTEILER",
			"ATTACK_LADEVORGANG"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM",
			"ATTACK_METALLKLAUE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM",
			"ATTACK_METALLKLAUE",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_METRONOM",
			"ATTACK_METALLKLAUE",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_PRUEGLER",
			"ATTACK_BLUTSAUGER",
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_AGILITAET",
			"ATTACK_BEZIRZER",
			"ATTACK_GIFTSCHWEIF",
			"ATTACK_HITZEWELLE"
		],
		[
			"ATTACK_PRUEGLER",
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_BLUTSAUGER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_FLAMMENBLITZ",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET",
			"ATTACK_BEZIRZER",
			"ATTACK_GIFTSCHWEIF",
			"ATTACK_HITZEWELLE",
			"ATTACK_DRACHENTANZ",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_SILBERHAUCH",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_KNIRSCHER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_METALLKLAUE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_STACHLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_METALLKLAUE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_MIMIKRY",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_TAUCHER"
		],
		[
			"ATTACK_SUPERZAHN",
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RISIKOTACKLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_ZORNKLINGE",
			"ATTACK_KOPFNUSS",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_NAHKAMPF"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_PSYSTRAHL",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_GRIMASSE",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_KOPFNUSS",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_GEGENSCHLAG"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_NOTSITUATION",
			"ATTACK_RECHTE_HAND",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_BAUCHTROMMEL",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_STAFETTE",
			"ATTACK_AGILITAET",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FELSWURF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_SPRUNGFEDER",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_SCHNARCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_ZORNKLINGE",
			"ATTACK_MIMIKRY",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KREIDESCHREI",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB"
		],
		[
			"ATTACK_GROLL",
			"ATTACK_NOTSITUATION",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_WUTANFALL",
			"ATTACK_MEGAKICK",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_EISHIEB",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_GEGENSCHLAG",
			"ATTACK_KONTER",
			"ATTACK_KREIDESCHREI",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_NOTSITUATION",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_STACHLER",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_LEIDTEILER",
			"ATTACK_SYNTHESE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_FELSWURF",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAMENBOMBEN",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_LEIDTEILER",
			"ATTACK_SYNTHESE"
		],
		[
			"ATTACK_KOPFNUSS",
			"ATTACK_SCHNARCHER",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_RISIKOTACKLE"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_FUSSKICK",
			"ATTACK_FADENSCHUSS",
			"ATTACK_LEHMSCHUSS",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KREIDESCHREI",
			"ATTACK_EISENABWEHR"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_NACHTNEBEL",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_VITALGLOCKE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_MEGAHIEB",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_ZORNKLINGE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_EISENABWEHR",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_ZORNKLINGE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_BODYSLAM",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDHOSE",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR",
			"ATTACK_DRACHENTANZ",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_FEUERWIRBEL",
			"ATTACK_HYDROPUMPE",
			"ATTACK_EISENABWEHR",
			"ATTACK_HITZEWELLE",
			"ATTACK_DRACHENTANZ",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[
			"ATTACK_EISENABWEHR",
			"ATTACK_KOPFNUSS"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_KOSMIK_KRAFT",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_KOSMIK_KRAFT",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FELSWURF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_EISSPEER",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_METALLSOUND",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EINIGLER",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_TAUCHER",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_EINIGLER"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_METALLKLAUE",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_KOSMIK_KRAFT",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_TAUCHER",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_KOPFNUSS",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_HYDROPUMPE"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_TRIPLETTE",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_TAUCHER",
			"ATTACK_TIEFSCHLAG",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_WINDSCHNITT",
			"ATTACK_BODYSLAM",
			"ATTACK_DIEBESKUSS",
			"ATTACK_STAFETTE",
			"ATTACK_AGILITAET",
			"ATTACK_WINDHOSE",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_BEZIRZER",
			"ATTACK_DRACHENTANZ"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_DRACO_METEOR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_WUTANFALL",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_METEOROLOGE",
			"ATTACK_TRIPLETTE",
			"ATTACK_ZORNKLINGE",
			"ATTACK_SEHER",
			"ATTACK_TAUCHER",
			"ATTACK_WHIRLPOOL",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYCHECK",
			"ATTACK_WINDSCHNITT",
			"ATTACK_STAFETTE",
			"ATTACK_WINDHOSE",
			"ATTACK_AGILITAET",
			"ATTACK_BODYSLAM"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_AURASPHAERE",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_NACHTMAHR",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_AUFRUHR",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_KNUDDLER",
			"ATTACK_BEGRENZER",
			"ATTACK_METALLSOUND",
			"ATTACK_PSYSTRAHL",
			"ATTACK_AMNESIE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_ZUGABE",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_STAFETTE",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_EISENABWEHR",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_NACHTMAHR",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_FUSSKICK",
			"ATTACK_BEGRENZER",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SEHER",
			"ATTACK_PSYSTRAHL",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_AGILITAET",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_LEIDTEILER",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_EISHIEB"
		],
		[
			"ATTACK_TRUGTRAENE",
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_CHARME",
			"ATTACK_NACHTMAHR",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_BEGRENZER",
			"ATTACK_KLAMMERGRIFF",
			"ATTACK_SAEUSELSTIMME",
			"ATTACK_PSYSTRAHL",
			"ATTACK_ZUGABE",
			"ATTACK_MIMIKRY",
			"ATTACK_ABGESANG",
			"ATTACK_STAFETTE",
			"ATTACK_DIEBESKUSS",
			"ATTACK_EINIGLER",
			"ATTACK_SCHALLWELLE"
		],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_RECHTE_HAND",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_KNIRSCHER",
			"ATTACK_AUFRUHR",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_STACHLER",
			"ATTACK_FELSWURF",
			"ATTACK_ZORNKLINGE",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_HITZEWELLE",
			"ATTACK_EINIGLER",
			"ATTACK_METALLKLAUE",
			"ATTACK_SANDGRAB",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_STERNSCHAUER",
			"ATTACK_SCHNARCHER",
			"ATTACK_RISIKOTACKLE",
			"ATTACK_MEGAKICK",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_WUCHTSCHLAG",
			"ATTACK_FUSSKICK",
			"ATTACK_ZORNKLINGE",
			"ATTACK_NASSSCHWEIF",
			"ATTACK_FEUERSCHLAG",
			"ATTACK_KOPFNUSS",
			"ATTACK_EISHIEB",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_KONTER",
			"ATTACK_DONNERSCHLAG",
			"ATTACK_ROLLENTAUSCH",
			"ATTACK_METRONOM",
			"ATTACK_EINIGLER",
			"ATTACK_MEGAHIEB"
		],
		[
			"ATTACK_GEOFISSUR",
			"ATTACK_GRIMASSE",
			"ATTACK_SCHNARCHER",
			"ATTACK_FINALE",
			"ATTACK_PSYCHO_PLUS",
			"ATTACK_GEDULD",
			"ATTACK_SCHAEDELWUMME",
			"ATTACK_ANTIK_KRAFT",
			"ATTACK_KOPFNUSS",
			"ATTACK_LEHMSCHELLE",
			"ATTACK_BODYCHECK",
			"ATTACK_MIMIKRY",
			"ATTACK_BODYSLAM",
			"ATTACK_WINDHOSE",
			"ATTACK_DRACHENTANZ"
		]
	]
}""")

In [92]:
from collections import Counter


cnt = Counter(
    move for xi in x["data"] for move in xi
)
cnt

Counter({'ATTACK_SCHNARCHER': 388,
         'ATTACK_MIMIKRY': 313,
         'ATTACK_KOPFNUSS': 260,
         'ATTACK_BODYSLAM': 260,
         'ATTACK_RISIKOTACKLE': 246,
         'ATTACK_LEHMSCHELLE': 234,
         'ATTACK_BODYCHECK': 221,
         'ATTACK_RECHTE_HAND': 215,
         'ATTACK_STERNSCHAUER': 188,
         'ATTACK_PSYCHO_PLUS': 130,
         'ATTACK_RASEREI': 118,
         'ATTACK_MEGAHIEB': 118,
         'ATTACK_MEGAKICK': 117,
         'ATTACK_AUFRUHR': 116,
         'ATTACK_GEDULD': 115,
         'ATTACK_WUCHTSCHLAG': 101,
         'ATTACK_KONTER': 97,
         'ATTACK_NOTSITUATION': 94,
         'ATTACK_EISHIEB': 91,
         'ATTACK_EINIGLER': 90,
         'ATTACK_DONNERSCHLAG': 90,
         'ATTACK_ZORNKLINGE': 86,
         'ATTACK_SCANNER': 84,
         'ATTACK_SCHAEDELWUMME': 81,
         'ATTACK_WHIRLPOOL': 81,
         'ATTACK_LEHMSCHUSS': 80,
         'ATTACK_FEUERSCHLAG': 76,
         'ATTACK_GROLL': 75,
         'ATTACK_METEOROLOGE': 73,
         'ATTACK_GRIM

In [91]:
x.keys()

dict_keys(['label', 'type', 'data'])